In [1]:
# CONVENIENCE: 
import sys as sys

print(sys.path)

# If running notebook outside of scrna_mvae directory, add it to the path
sys.path.append('/content/CellUntangler/')
print(sys.version)

import os

import torch

from src.data.umi_data import UMIVaeDataset
from src.celluntangler import utils
from src.celluntangler.models import Trainer
from src.celluntangler.models.nb_vae import NBVAE
import numpy as np

import pandas as pd
import scanpy as sc

""" 
hela_dataset_path = "/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/data/hela/hela_select_with_phase.h5ad"
adata = sc.read_h5ad(hela_dataset_path)
""" 
adata = sc.read_loom("/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/code/liver_time.loom")
adata.var["gene_symbols"] = adata.var.index

# NB: no longer actually cell cycle genes! should be zonation genes instead
cell_cycle_genes_path = "/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/code/CellUntangler/genes/zonation_genes_mouse.tsv"
cell_cycle_genes = pd.read_csv(cell_cycle_genes_path, header=None, sep="\t")

# Genes involved with the cell cycle present in adata
contained_genes = adata.var["gene_symbols"].isin(cell_cycle_genes[0])
print(f"Number of cell cycle genes present in adata: {np.sum(contained_genes)}")
cycle_gene_indices = np.where(contained_genes)[0]
non_cycle_gene_indices = np.where(~contained_genes)[0]
rearranged_indices = np.hstack((cycle_gene_indices, non_cycle_gene_indices))
adata = adata[:, rearranged_indices]
adata.uns["new_gene_ordering"] = rearranged_indices

from src.celluntangler.models.get_config import get_config
config = get_config()
config.model_name = "e2, e10"
config.seed = 68715
config.init = "custom"
adata.X.todense()

x = adata.X.todense().astype(np.double)
batch = (np.zeros((x.shape[0], 1)) * -1).astype(np.int64)
# y holds the batch vector for the dataset
y = batch

# Create the dataset and separate it into training and test sets
in_dim = x.shape[1]
print(f"in_dim = {in_dim}")
batch_size = config.batch_size
dataset = UMIVaeDataset(batch_size=batch_size, in_dim=in_dim)
print(f"dataset.in_dim = {dataset.in_dim}")
# Create the dataset loaders
train_loader = dataset.create_loaders(x, y, seed=config.seed)

# A mask that is 1 where the gene is a cell cycling gene and 0 otherwise
mask_cyc = np.zeros(adata.n_vars)
mask_cyc[adata.var["gene_symbols"].isin(cell_cycle_genes[0])] = 1

# A mask that is 0 where the gene is a cell cycling gene and 1 otherwise
mask_all = np.ones(adata.n_vars)
mask_all[adata.var["gene_symbols"].isin(cell_cycle_genes[0])] = 0

mask = torch.tensor([mask_cyc, mask_all])

'''
'''

['/opt/homebrew/Cellar/python@3.11/3.11.12/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.12/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.12/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/code/venvs/CellUntangler/lib/python3.11/site-packages']
3.11.12 (main, Apr  8 2025, 14:15:29) [Clang 16.0.0 (clang-1600.0.26.6)]


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Number of cell cycle genes present in adata: 14


/var/folders/2z/f9h5sjqj6m9dz0ljc7yy_plw0000gn/T/ipykernel_45831/756976404.py:41: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["new_gene_ordering"] = rearranged_indices


in_dim = 14811
dataset.in_dim = 14811
Dataset seed: 68715


/var/folders/2z/f9h5sjqj6m9dz0ljc7yy_plw0000gn/T/ipykernel_45831/756976404.py:72: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  mask = torch.tensor([mask_cyc, mask_all])


'\n'

## Train CellUntangler.

Specify whether to use CPU or GPU.

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
use_gpu = False
if use_gpu:
  print("Using GPU")
  config.device = torch.device("cuda")
else:
  print("Using CPU")
  config.device = torch.device("cpu")

Using CPU


Below is to save embeddings generated through the epochs. Set visualization_information = None if it is not desired.

In [11]:
# The path to save the intermediate embeddings to
epoch_embeddings_save_path = "./experiments/liver_zonation_repro"
visualize_information={}
# The epochs to save the intermediate embeddings for
visualize_information["epochs"]=[i for i in range(0, 500, 50)] 
visualize_information["x"]=x
visualize_information["y"]=y
visualize_information["embeddings_save_path"]=epoch_embeddings_save_path
visualize_information["model_name"]=config.model_name
visualize_information["device"]=config.device

In [12]:
adata.obs

,RNA_snn_res.0.8,nCount_RNA,nFeature_RNA,orig.ident,percent.mt,rep,seurat_clusters,time,zonation1
CellID,,,,,,,,,
ZT00B1,3,3287.0,1279,1,15.150593,B,3,T00,0.285488
ZT00B4,7,1397.0,646,1,13.242663,B,7,T00,0.539859
ZT00B5,7,1293.0,615,1,13.689095,B,7,T00,0.485066
ZT00B6,3,1339.0,606,1,21.657954,B,3,T00,0.566012
ZT00B7,3,1712.0,736,1,18.574766,B,3,T00,0.437602
...,...,...,...,...,...,...,...,...,...
ZT18B1782,5,1394.0,716,1,16.427547,B,5,T18,0.745925
ZT18B1783,2,1523.0,691,1,22.127380,B,2,T18,0.444957
ZT18B1784,5,1677.0,786,1,19.260584,B,5,T18,0.500790


In [13]:
mask.shape

torch.Size([2, 14811])

In [14]:
# TODO - train for 500 epochs once verified working!
config.max_epochs = 100
config.epochs = 100 

In [ ]:
if config.seed:
    print(config.seed)
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)
    np.random.default_rng(config.seed)

model_name = config.model_name
components = utils.parse_components(model_name, config.fixed_curvature)
# =====
model = NBVAE(h_dim=config.h_dim,
              components=components,
              mask=mask,
              dataset=dataset,
              config=config,
              component_subspaces=None
                  ).to(config.device)

trainer = Trainer(model)

optimizer = trainer.build_optimizer(learning_rate=config.learning_rate,
                                        fixed_curvature=config.fixed_curvature,
                                        use_adamw=config.use_adamw,
                                        weight_decay=config.weight_decay)

betas = utils.linear_betas(config.start,
                           config.end,
                           end_epoch=config.end_epoch,
                           epochs=config.epochs)

trainer.train_epochs(optimizer=optimizer,
                       train_data=train_loader,
                       betas=betas,
                       likelihood_n=0,
                       max_epochs=config.max_epochs,
                           visualize_information=visualize_information)

68715
Linear(in_features=32, out_features=2, bias=True)
initializing Xavier uniform weights in Linear
Linear(in_features=32, out_features=2, bias=True)
initializing Xavier uniform weights in Linear
EuclideanComponent(R^2)
	TrainEpoch 0:	

/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/code/venvs/CellUntangler/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'bce': -4115.331, 'kl': 40.071, 'elbo': -4155.402, 'll': 0.0, 'mi': 0.0, 'cov_norm': 0.0, 'beta': np.float64(1.0), 'comp_000_e2/curvature': 0.0, 'comp_001_e10/curvature': 0.0}
	TrainEpoch 1:	{'bce': -3424.266, 'kl': 13.774, 'elbo': -3438.04, 'll': 0.0, 'mi': 0.0, 'cov_norm': 0.0, 'beta': np.float64(1.0), 'comp_000_e2/curvature': 0.0, 'comp_001_e10/curvature': 0.0}
	TrainEpoch 2:	{'bce': -3398.074, 'kl': 12.793, 'elbo': -3410.867, 'll': 0.0, 'mi': 0.0, 'cov_norm': 0.0, 'beta': np.float64(1.0), 'comp_000_e2/curvature': 0.0, 'comp_001_e10/curvature': 0.0}
	TrainEpoch 3:	{'bce': -3390.459, 'kl': 11.399, 'elbo': -3401.858, 'll': 0.0, 'mi': 0.0, 'cov_norm': 0.0, 'beta': np.float64(1.0), 'comp_000_e2/curvature': 0.0, 'comp_001_e10/curvature': 0.0}
	TrainEpoch 4:	{'bce': -3385.674, 'kl': 10.01, 'elbo': -3395.683, 'll': 0.0, 'mi': 0.0, 'cov_norm': 0.0, 'beta': np.float64(1.0), 'comp_000_e2/curvature': 0.0, 'comp_001_e10/curvature': 0.0}
	TrainEpoch 5:	{'bce': -3382.51, 'kl': 8.86, 'elbo': -339

## Analysis

In [ ]:
embeddings_save_path = "./experiments/liver_zonation_repro"
a = trainer.model(torch.log1p(torch.tensor(x, device=config.device)), torch.tensor(y, device=config.device))
np.savetxt(os.path.join(embeddings_save_path, f'{model_name}_all_encode_v63_z_params.txt'), a[4].detach().to(torch.device("cpu")).numpy())

In [ ]:
gene_expression_matrix_path = "./experiments/liver_zonation_repro"

In [ ]:
import torch.nn as nn
# Encode the batch vector which must also be passed to the decoder
batch = nn.functional.one_hot(torch.tensor(y[:,0]), 1).to(config.device)

library_size = torch.sum(torch.tensor(x), dim=1).to(config.device)

In [ ]:
# To enhance zonation, mask the non-zonation latent representation and pass it to the decoder
# The latent representations have a dimension of 12 (2 dimensions for zonation + 10 dimensions for non-zonation)
z1_mask = torch.zeros(12, device=config.device)
z1_mask[0:2] = 1

z_params = a[4].detach()
x_z1_params_mu = trainer.model.decode(z_params*z1_mask, batch)[0]
x_z1_params_mu = x_z1_params_mu * library_size[:, None] 
# Q: Why does the library_size (RNA expression of each cell (?)) need to be considered 
#    when re-deriving the latent variable values of z1? 
np.savetxt(os.path.join(gene_expression_matrix_path,"x_z1_params_mu.txt"), x_z1_params_mu.detach().to(torch.device("cpu")).numpy())

In [ ]:
# To filter for zonation, mask the zonation latent representation and pass it to the decoder
z2_mask = torch.zeros(12, device=config.device)
z2_mask[2:] = 1

z_params = a[4].detach()
x_z2_params_mu = trainer.model.decode(z_params*z2_mask, batch)[0]
x_z2_params_mu = x_z2_params_mu * library_size[:, None]
np.savetxt(os.path.join(gene_expression_matrix_path,"x_z2_params_mu.txt"), x_z2_params_mu.detach().to(torch.device("cpu")).numpy())

In [ ]:
import matplotlib.pyplot as plt

from src.visualization.helpers import split_embeddings
from src.visualization.visualization_functions import visualize_poincare_from_lorentz, compute_umap

In [ ]:
embeddings_save_path = "./experiments/liver_zonation_repro"
model_name = "e2, e10"
# Load in the embeddings
# Q: What does "v63" mean in the context of the below? 
embeddings = np.loadtxt(os.path.join(embeddings_save_path, f'{model_name}_all_encode_v63_z_params.txt'))

In [ ]:
# Returns a dictionary where the keys are of the format componenti_subspace
# E.g., e2, e10 would result in keys of component1_e2 and component2_e10
# The values are the corresponding embeddings
component_embeddings = split_embeddings(model_name, embeddings)
for c in component_embeddings:
    adata.obsm[c] = component_embeddings[c]

In [ ]:
compute_umap(adata,
             l_neighbors=50,             # Q: What is this paraeter doing?
             color=['zonation1','time'],
             n_pcs=None, 
             embeddings_key="component1_e2",
             use_original_umap=False,
             palette=None,
             additional_save_information=[],
             title=None,
             save_figure=False)

In [ ]:
compute_umap(adata,
             l_neighbors=50,             # Q: What is this paraeter doing?
             color=['zonation1','time'],
             n_pcs=None, 
             embeddings_key="component2_e10",
             use_original_umap=False,
             palette=None,
             additional_save_information=[],
             title=None,
             save_figure=False)